In [1]:
import os
import sys
import shutil
import subprocess
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 0. Biopython
# ============================================================

if importlib.util.find_spec("Bio") is None:
    print("Biopython not found. Installing into current Jupyter kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "biopython"])

from Bio import Entrez, SeqIO


# ============================================================
# 1. CONFIG
# ============================================================

# NCBI requires an email for Entrez requests.
NCBI_EMAIL = "your_email@example.com"  # replace with your real email

DATASET_TAG = "mtDNA15_heterogeneous_primate"

OUTDIR = Path(f"{DATASET_TAG}_MAFFT_distances")
OUTDIR.mkdir(exist_ok=True)

RAW_FASTA = OUTDIR / f"{DATASET_TAG}_raw.fasta"
CLEAN_FASTA = OUTDIR / f"{DATASET_TAG}_clean.fasta"
ALIGNED_FASTA = OUTDIR / f"{DATASET_TAG}_aligned_mafft.fasta"
MAFFT_LOG = OUTDIR / "mafft.stderr.log"

OUT_LABEL_MAP = OUTDIR / "T_labels_mapping.csv"

OUT_PDIST_T = OUTDIR / "Dref_MAFFT_pairwise_deletion_pdistance_Tlabels.csv"
OUT_JC69_T = OUTDIR / "Dref_MAFFT_pairwise_deletion_JC69_Tlabels.csv"
OUT_K80_T = OUTDIR / "Dref_MAFFT_pairwise_deletion_K80_Tlabels.csv"

OUT_PDIST_FULL = OUTDIR / "Dref_MAFFT_pairwise_deletion_pdistance_full_labels.csv"
OUT_JC69_FULL = OUTDIR / "Dref_MAFFT_pairwise_deletion_JC69_full_labels.csv"
OUT_K80_FULL = OUTDIR / "Dref_MAFFT_pairwise_deletion_K80_full_labels.csv"

OUT_COUNTS_VALID = OUTDIR / "pairwise_valid_sites_Tlabels.csv"
OUT_COUNTS_MISMATCHES = OUTDIR / "pairwise_mismatches_Tlabels.csv"
OUT_COUNTS_TRANSITIONS = OUTDIR / "pairwise_transitions_Tlabels.csv"
OUT_COUNTS_TRANSVERSIONS = OUTDIR / "pairwise_transversions_Tlabels.csv"

OUT_SUMMARY = OUTDIR / "heterogeneous_dataset_summary.csv"

# If MAFFT is not in PATH, set full path manually, e.g. "/usr/bin/mafft"
MAFFT_EXE = shutil.which("mafft") or "/usr/bin/mafft"

# Set True if you want to redownload / realign even if files already exist.
FORCE_DOWNLOAD = False
FORCE_MAFFT = True


# ============================================================
# 1.1 HETEROGENEOUS 15-TAXON PRIMATE DATASET
# ============================================================

# Accessions are written without version suffixes.
# The code below removes version suffixes from downloaded FASTA IDs automatically.
#
# Purpose: same matrix size as the main 15x15 experiment,
# but taxonomically broader than the original Cercopithecidae-only set.

TAXA = [
    # Hominoids
    {"accession": "NC_012920", "species": "Homo_sapiens",        "group": "Hominoidea"},
    {"accession": "NC_001643", "species": "Pan_troglodytes",     "group": "Hominoidea"},
    {"accession": "NC_001644", "species": "Pan_paniscus",        "group": "Hominoidea"},
    {"accession": "D38114",    "species": "Gorilla_gorilla",     "group": "Hominoidea"},
    {"accession": "NC_002083", "species": "Pongo_abelii",        "group": "Hominoidea"},
    {"accession": "NC_002082", "species": "Hylobates_lar",       "group": "Hominoidea"},

    # Old World monkeys
    {"accession": "NC_005943", "species": "Macaca_mulatta",      "group": "Old_World_monkeys"},
    {"accession": "NC_001992", "species": "Papio_hamadryas",     "group": "Old_World_monkeys"},
    {"accession": "NC_007009", "species": "Chlorocebus_aethiops", "group": "Old_World_monkeys"},
    {"accession": "NC_006901", "species": "Colobus_guereza",     "group": "Old_World_monkeys"},

    # New World monkeys
    {"accession": "NC_002763", "species": "Cebus_albifrons",     "group": "New_World_monkeys"},
    {"accession": "KM588314",  "species": "Callithrix_jacchus",  "group": "New_World_monkeys"},

    # More distant primate lineages
    {"accession": "NC_002811", "species": "Tarsius_bancanus",    "group": "Tarsiiformes"},
    {"accession": "NC_002765", "species": "Nycticebus_coucang",  "group": "Strepsirrhini"},
    {"accession": "NC_004025", "species": "Lemur_catta",         "group": "Strepsirrhini"},
]

ACCESSIONS = {row["accession"]: row["species"] for row in TAXA}
GROUPS = {row["accession"]: row["group"] for row in TAXA}

FULL_LABELS = [f"{row['species']}_{row['accession']}" for row in TAXA]
T_LABELS = [f"T{i + 1}" for i in range(len(TAXA))]


# ============================================================
# 2. HELPERS
# ============================================================

def run_command(cmd, stdout_path=None, stderr_path=None):
    print("\nRunning:")
    print(" ".join(map(str, cmd)))

    stdout_handle = open(stdout_path, "w", encoding="utf-8") if stdout_path else None
    stderr_handle = open(stderr_path, "w", encoding="utf-8") if stderr_path else None

    try:
        result = subprocess.run(
            list(map(str, cmd)),
            stdout=stdout_handle if stdout_handle else subprocess.PIPE,
            stderr=stderr_handle if stderr_handle else subprocess.PIPE,
            text=True,
            check=False,
        )

        if result.returncode != 0:
            if result.stderr:
                print(result.stderr)
            raise RuntimeError(f"Command failed with exit code {result.returncode}")

        if result.stdout and not stdout_path:
            print(result.stdout[:1000])

    finally:
        if stdout_handle:
            stdout_handle.close()
        if stderr_handle:
            stderr_handle.close()


def accession_key(record_id: str) -> str:
    """
    Extract accession without version suffix.

    Handles common FASTA ID forms:
        NC_012920.1
        D38114.1
        gi|251831106|ref|NC_012920.1|
    """
    rid = record_id.strip().split()[0]

    if "|" in rid:
        parts = [p for p in rid.split("|") if p]
        for p in parts:
            p0 = p.split(".")[0]
            if p0 in ACCESSIONS:
                return p0

    return rid.split(".")[0]


def download_fasta(accessions, output_fasta):
    Entrez.email = NCBI_EMAIL
    ids = ",".join(accessions)

    print(f"Downloading {len(accessions)} sequences from NCBI...")
    with Entrez.efetch(
        db="nucleotide",
        id=ids,
        rettype="fasta",
        retmode="text",
    ) as handle:
        fasta_text = handle.read()

    if not fasta_text.strip():
        raise RuntimeError("NCBI returned empty FASTA text.")

    output_fasta.write_text(fasta_text, encoding="utf-8")
    print(f"Saved raw FASTA: {output_fasta}")


def sanitize_fasta(raw_fasta, clean_fasta):
    records = list(SeqIO.parse(raw_fasta, "fasta"))
    if not records:
        raise ValueError("No records found in raw FASTA.")

    found = set()
    clean_records_by_acc = {}

    for rec in records:
        acc = accession_key(rec.id)

        if acc not in ACCESSIONS:
            raise ValueError(
                f"Unexpected accession in downloaded FASTA: {rec.id}\n"
                f"Parsed accession key: {acc}\n"
                f"Expected one of: {sorted(ACCESSIONS.keys())}"
            )

        label = f"{ACCESSIONS[acc]}_{acc}"

        rec.id = label
        rec.name = label
        rec.description = ""

        clean_records_by_acc[acc] = rec
        found.add(acc)

    missing = set(ACCESSIONS) - found
    if missing:
        raise ValueError(f"Missing accessions in downloaded FASTA: {sorted(missing)}")

    extra = found - set(ACCESSIONS)
    if extra:
        raise ValueError(f"Unexpected extra accessions: {sorted(extra)}")

    # Reorder records exactly according to TAXA / ACCESSIONS order.
    # Do NOT use rec.id.split('_')[-1], because RefSeq accessions contain underscores.
    ordered_records = [clean_records_by_acc[row["accession"]] for row in TAXA]

    SeqIO.write(ordered_records, clean_fasta, "fasta")
    print(f"Saved clean FASTA: {clean_fasta}")

    print("\nClean FASTA order:")
    for i, rec in enumerate(ordered_records, 1):
        print(f"{i:2d}. {rec.id}  length={len(rec.seq)}")


def read_alignment(aligned_fasta, expected_labels=None):
    records = list(SeqIO.parse(aligned_fasta, "fasta"))
    if not records:
        raise ValueError("No records found in aligned FASTA.")

    if expected_labels is not None:
        by_id = {rec.id: rec for rec in records}
        missing = set(expected_labels) - set(by_id)
        extra = set(by_id) - set(expected_labels)

        if missing:
            raise ValueError(f"Aligned FASTA is missing labels: {sorted(missing)}")
        if extra:
            raise ValueError(f"Aligned FASTA has unexpected labels: {sorted(extra)}")

        records = [by_id[label] for label in expected_labels]

    labels = [rec.id for rec in records]
    seqs = [str(rec.seq).upper() for rec in records]

    lengths = {len(s) for s in seqs}
    if len(lengths) != 1:
        raise ValueError(f"Aligned sequences have different lengths: {lengths}")

    return labels, seqs


def pairwise_deletion_counts(seq1, seq2):
    """
    Pairwise deletion:
    use only columns where both sequences have A/C/G/T.
    Gaps, N, ambiguity codes are excluded pairwise.
    """
    valid_bases = {"A", "C", "G", "T"}

    valid = 0
    mismatches = 0
    transitions = 0
    transversions = 0

    transition_pairs = {
        ("A", "G"), ("G", "A"),
        ("C", "T"), ("T", "C"),
    }

    for a, b in zip(seq1, seq2):
        if a not in valid_bases or b not in valid_bases:
            continue

        valid += 1

        if a != b:
            mismatches += 1

            if (a, b) in transition_pairs:
                transitions += 1
            else:
                transversions += 1

    return valid, mismatches, transitions, transversions


def jc69_distance(p):
    """
    Jukes-Cantor correction:
        d = -3/4 log(1 - 4p/3)

    Returns NaN if p is outside the valid JC69 correction range.
    """
    if not np.isfinite(p):
        return np.nan
    if p < 0:
        return np.nan
    if p == 0:
        return 0.0

    arg = 1.0 - 4.0 * p / 3.0
    if arg <= 0:
        return np.nan

    return float(-0.75 * np.log(arg))


def k80_distance(P, Q):
    """
    Kimura 2-parameter correction:
        P = transition proportion
        Q = transversion proportion
        d = -1/2 log(1 - 2P - Q) - 1/4 log(1 - 2Q)

    Returns NaN if P,Q are outside the valid K80 correction range.
    """
    if not np.isfinite(P) or not np.isfinite(Q):
        return np.nan

    a = 1.0 - 2.0 * P - Q
    b = 1.0 - 2.0 * Q

    if a <= 0 or b <= 0:
        return np.nan

    return float(-0.5 * np.log(a) - 0.25 * np.log(b))


def compute_distance_matrices(labels, seqs):
    n = len(seqs)

    D_p = np.zeros((n, n), dtype=float)
    D_jc = np.zeros((n, n), dtype=float)
    D_k80 = np.zeros((n, n), dtype=float)

    N_valid = np.zeros((n, n), dtype=int)
    N_mismatch = np.zeros((n, n), dtype=int)
    N_transition = np.zeros((n, n), dtype=int)
    N_transversion = np.zeros((n, n), dtype=int)

    for i in range(n):
        for j in range(i + 1, n):
            valid, mismatches, transitions, transversions = pairwise_deletion_counts(
                seqs[i], seqs[j]
            )

            N_valid[i, j] = N_valid[j, i] = valid
            N_mismatch[i, j] = N_mismatch[j, i] = mismatches
            N_transition[i, j] = N_transition[j, i] = transitions
            N_transversion[i, j] = N_transversion[j, i] = transversions

            if valid == 0:
                p = np.nan
                P = np.nan
                Q = np.nan
            else:
                p = mismatches / valid
                P = transitions / valid
                Q = transversions / valid

            D_p[i, j] = D_p[j, i] = p
            D_jc[i, j] = D_jc[j, i] = jc69_distance(p)
            D_k80[i, j] = D_k80[j, i] = k80_distance(P, Q)

    np.fill_diagonal(D_p, 0.0)
    np.fill_diagonal(D_jc, 0.0)
    np.fill_diagonal(D_k80, 0.0)

    counts = {
        "valid_sites": N_valid,
        "mismatches": N_mismatch,
        "transitions": N_transition,
        "transversions": N_transversion,
    }

    return D_p, D_jc, D_k80, counts


def save_matrix(M, labels, path):
    df = pd.DataFrame(M, index=labels, columns=labels)
    df.to_csv(path)
    print(f"Saved: {path}")


def matrix_summary_dict(name, M):
    off = M[np.triu_indices_from(M, k=1)]
    return {
        "name": name,
        "min": float(np.nanmin(off)),
        "median": float(np.nanmedian(off)),
        "mean": float(np.nanmean(off)),
        "max": float(np.nanmax(off)),
        "std": float(np.nanstd(off)),
        "nan_count": int(np.isnan(off).sum()),
        "n_pairs": int(off.size),
    }


def print_matrix_summary(name, M):
    s = matrix_summary_dict(name, M)
    print(f"\n{name}")
    print(f"  min:      {s['min']:.6f}")
    print(f"  median:   {s['median']:.6f}")
    print(f"  mean:     {s['mean']:.6f}")
    print(f"  max:      {s['max']:.6f}")
    print(f"  std:      {s['std']:.6f}")
    print(f"  nan:      {s['nan_count']}")
    print(f"  n_pairs:  {s['n_pairs']}")


def robust_triplet_delta(D, omega=2.0, eps=1e-12):
    """
    Optional diagnostic: robust triplet ultrametric-violation score.

    This follows the same idea as the manuscript:
    for sorted sides a >= b >= c,
    if triangle inequality is violated, use a large distance-based penalty;
    otherwise compute angles and use (alpha - beta) / gamma.

    Returns:
        total_delta, mean_delta_per_triplet, n_triplets
    """
    D = np.asarray(D, dtype=float)
    n = D.shape[0]

    total = 0.0
    n_triplets = 0

    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                sides = np.array([D[i, j], D[i, k], D[j, k]], dtype=float)

                if np.any(~np.isfinite(sides)):
                    continue

                c, b, a = np.sort(sides)  # ascending, so a is largest

                if a >= b + c:
                    denom = max(b + c, eps)
                    delta = max(a / denom, omega)
                else:
                    # Angles opposite a,b,c by law of cosines.
                    cos_alpha = (b * b + c * c - a * a) / max(2.0 * b * c, eps)
                    cos_beta = (a * a + c * c - b * b) / max(2.0 * a * c, eps)
                    cos_gamma = (a * a + b * b - c * c) / max(2.0 * a * b, eps)

                    cos_alpha = np.clip(cos_alpha, -1.0, 1.0)
                    cos_beta = np.clip(cos_beta, -1.0, 1.0)
                    cos_gamma = np.clip(cos_gamma, -1.0, 1.0)

                    alpha = np.arccos(cos_alpha)
                    beta = np.arccos(cos_beta)
                    gamma = np.arccos(cos_gamma)

                    delta = (alpha - beta) / max(gamma, eps)

                total += float(delta)
                n_triplets += 1

    mean_delta = total / n_triplets if n_triplets else np.nan
    return total, mean_delta, n_triplets


def save_dataset_summary(D_p, D_jc, D_k80, counts, out_path):
    rows = []

    for name, M in [
        ("p_distance_pairwise_deletion", D_p),
        ("JC69_pairwise_deletion", D_jc),
        ("K80_pairwise_deletion", D_k80),
    ]:
        rows.append(matrix_summary_dict(name, M))

    valid_off = counts["valid_sites"][np.triu_indices_from(counts["valid_sites"], k=1)]

    rows.append({
        "name": "pairwise_valid_sites",
        "min": float(np.min(valid_off)),
        "median": float(np.median(valid_off)),
        "mean": float(np.mean(valid_off)),
        "max": float(np.max(valid_off)),
        "std": float(np.std(valid_off)),
        "nan_count": 0,
        "n_pairs": int(valid_off.size),
    })

    total_delta, mean_delta, n_triplets = robust_triplet_delta(D_p)
    rows.append({
        "name": "Delta_p_distance_total",
        "min": np.nan,
        "median": np.nan,
        "mean": float(total_delta),
        "max": np.nan,
        "std": np.nan,
        "nan_count": 0,
        "n_pairs": int(n_triplets),
    })
    rows.append({
        "name": "Delta_p_distance_per_triplet",
        "min": np.nan,
        "median": np.nan,
        "mean": float(mean_delta),
        "max": np.nan,
        "std": np.nan,
        "nan_count": 0,
        "n_pairs": int(n_triplets),
    })

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")


# ============================================================
# 3. DOWNLOAD / CLEAN FASTA
# ============================================================

if RAW_FASTA.exists() and not FORCE_DOWNLOAD:
    print(f"Raw FASTA already exists: {RAW_FASTA}")
else:
    download_fasta(list(ACCESSIONS.keys()), RAW_FASTA)

sanitize_fasta(RAW_FASTA, CLEAN_FASTA)


# ============================================================
# 4. MAFFT ALIGNMENT
# ============================================================

if not Path(MAFFT_EXE).exists() and shutil.which(MAFFT_EXE) is None:
    raise FileNotFoundError(
        f"Cannot find MAFFT executable: {MAFFT_EXE}\n"
        f"In WSL install it with: sudo apt install mafft"
    )

if ALIGNED_FASTA.exists() and not FORCE_MAFFT:
    print(f"MAFFT alignment already exists: {ALIGNED_FASTA}")
else:
    mafft_cmd = [
        MAFFT_EXE,
        "--auto",
        str(CLEAN_FASTA),
    ]

    run_command(
        mafft_cmd,
        stdout_path=ALIGNED_FASTA,
        stderr_path=MAFFT_LOG,
    )

    print(f"Saved MAFFT alignment: {ALIGNED_FASTA}")


# ============================================================
# 5. COMPUTE DISTANCE MATRICES
# ============================================================

labels_alignment, seqs = read_alignment(ALIGNED_FASTA, expected_labels=FULL_LABELS)

print("\nAlignment summary:")
print(f"  sequences: {len(seqs)}")
print(f"  columns:   {len(seqs[0])}")

print("\nAlignment labels:")
for i, lab in enumerate(labels_alignment, 1):
    print(f"{i:2d}. {lab}")

D_p, D_jc, D_k80, counts = compute_distance_matrices(labels_alignment, seqs)


# ============================================================
# 6. SAVE LABEL MAP
# ============================================================

label_map_df = pd.DataFrame({
    "T_label": T_LABELS,
    "full_label": FULL_LABELS,
    "species": [row["species"] for row in TAXA],
    "accession": [row["accession"] for row in TAXA],
    "group": [row["group"] for row in TAXA],
})

label_map_df.to_csv(OUT_LABEL_MAP, index=False)
print(f"\nSaved: {OUT_LABEL_MAP}")


# ============================================================
# 7. SAVE MATRICES
# ============================================================

# T-label matrices for your completion pipeline.
save_matrix(D_p, T_LABELS, OUT_PDIST_T)
save_matrix(D_jc, T_LABELS, OUT_JC69_T)
save_matrix(D_k80, T_LABELS, OUT_K80_T)

# Full-label matrices for manuscript transparency.
save_matrix(D_p, FULL_LABELS, OUT_PDIST_FULL)
save_matrix(D_jc, FULL_LABELS, OUT_JC69_FULL)
save_matrix(D_k80, FULL_LABELS, OUT_K80_FULL)

# Counts matrices.
save_matrix(counts["valid_sites"], T_LABELS, OUT_COUNTS_VALID)
save_matrix(counts["mismatches"], T_LABELS, OUT_COUNTS_MISMATCHES)
save_matrix(counts["transitions"], T_LABELS, OUT_COUNTS_TRANSITIONS)
save_matrix(counts["transversions"], T_LABELS, OUT_COUNTS_TRANSVERSIONS)


# ============================================================
# 8. QUICK DIAGNOSTICS
# ============================================================

print_matrix_summary("p-distance, pairwise deletion", D_p)
print_matrix_summary("JC69 corrected distance", D_jc)
print_matrix_summary("K80 corrected distance", D_k80)

valid_off = counts["valid_sites"][np.triu_indices_from(counts["valid_sites"], k=1)]

print("\nPairwise valid sites after gap/ambiguity deletion:")
print(f"  min:    {valid_off.min()}")
print(f"  median: {np.median(valid_off):.0f}")
print(f"  mean:   {valid_off.mean():.1f}")
print(f"  max:    {valid_off.max()}")

delta_total, delta_mean, n_triplets = robust_triplet_delta(D_p)
print("\nTriplet ultrametric diagnostic on p-distance matrix:")
print(f"  Delta total:       {delta_total:.6f}")
print(f"  Delta per triplet: {delta_mean:.6f}")
print(f"  triplets:          {n_triplets}")

save_dataset_summary(D_p, D_jc, D_k80, counts, OUT_SUMMARY)

print("\nRecommended main D_ref file for completion benchmark:")
print(f"  p-distance: {OUT_PDIST_T}")

print("\nOptional corrected-distance files:")
print(f"  JC69: {OUT_JC69_T}")
print(f"  K80:  {OUT_K80_T}")

print("\nDone.")

Saved raw FASTA: mtDNA15_heterogeneous_primate_MAFFT_distances/mtDNA15_heterogeneous_primate_raw.fasta
Saved clean FASTA: mtDNA15_heterogeneous_primate_MAFFT_distances/mtDNA15_heterogeneous_primate_clean.fasta

Clean FASTA order:
 1. Homo_sapiens_NC_012920  length=16569
 2. Pan_troglodytes_NC_001643  length=16554
 3. Pan_paniscus_NC_001644  length=16563
 4. Gorilla_gorilla_D38114  length=16364
 5. Pongo_abelii_NC_002083  length=16499
 6. Hylobates_lar_NC_002082  length=16472
 7. Macaca_mulatta_NC_005943  length=16564
 8. Papio_hamadryas_NC_001992  length=16521
 9. Chlorocebus_aethiops_NC_007009  length=16389
10. Colobus_guereza_NC_006901  length=16648
11. Cebus_albifrons_NC_002763  length=16554
12. Callithrix_jacchus_KM588314  length=16499
13. Tarsius_bancanus_NC_002811  length=16927
14. Nycticebus_coucang_NC_002765  length=16764
15. Lemur_catta_NC_004025  length=17036

Running:
/usr/bin/mafft --auto mtDNA15_heterogeneous_primate_MAFFT_distances/mtDNA15_heterogeneous_primate_clean.fast